# 🏦 Loan Approval Prediction System
---
| Field | Details |
|---|---|
| **Internship** | AIML Summer Internship 2026 |
| **Institute** | IIHMF, MNNIT Allahabad, Prayagraj |
| **Project** | Project 3 — Loan Approval Prediction System |
| **Team Member 1** | [Your Name] |
| **Team Member 2** | [Partner Name] |
| **Date** | June 2026 |

---
## 🎯 Objective
Develop a machine learning system to predict whether a loan application should be **approved or rejected** based on applicant information, using **6 classification models** (including XGBoost) with full preprocessing, EDA, feature engineering, and a Streamlit deployment built on the **Lumina Predict** design system.

## 📥 Cell 2 — Dataset Download from Kaggle

In [ ]:
# Step 1: Upload the dataset zip file
from google.colab import files
import os, zipfile, glob

print("="*60)
print("ACTION REQUIRED: Upload your dataset zip file")
print("File to upload: archive__1_.zip  (provided with this project)")
print("="*60)
uploaded = files.upload()  # Upload archive__1_.zip when prompted

# Step 2: Extract the zip
zip_name = list(uploaded.keys())[0]
os.makedirs('loan_data', exist_ok=True)
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('loan_data')
    print(f"\n✅ Extracted '{zip_name}' → loan_data/")

# Step 3: Confirm files
print("\n📁 Files extracted:")
for f in os.listdir('loan_data'):
    size = os.path.getsize(os.path.join('loan_data', f))
    print(f"   → {f}  ({size:,} bytes)")


## 📦 Cell 3 — Install & Import Libraries

In [ ]:
%%time
# Step 1: Install required libraries
!pip install imbalanced-learn xgboost -q

# Step 2: Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import json
import joblib
warnings.filterwarnings('ignore')

# Step 3: Scikit-learn
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, classification_report, confusion_matrix
)

# Step 4: Imbalanced-learn
from imblearn.over_sampling import SMOTE

# Step 5: XGBoost (now a core model, not optional)
from xgboost import XGBClassifier

# Step 6: Plot style
try:
    plt.style.use('seaborn-v0_8-whitegrid')
except:
    plt.style.use('seaborn-whitegrid')

print("\n✅ All libraries imported successfully!")
print(f"   Pandas: {pd.__version__} | NumPy: {np.__version__}")
print("   Models ready: Logistic Regression, Decision Tree, Random Forest, SVM, KNN, XGBoost")

## 📂 Cell 4 — Load Dataset

In [ ]:
# Step 1: Auto-detect and load training CSV
import glob

train_candidates = sorted(
    glob.glob('loan_data/train*.csv') +
    glob.glob('loan_data/**/train*.csv', recursive=True)
)
if not train_candidates:
    all_csvs = glob.glob('loan_data/**/*.csv', recursive=True)
    print("All CSVs found:", all_csvs)
    raise FileNotFoundError("Train CSV not found — check the file list above.")

TRAIN_CSV_PATH = train_candidates[0]
print(f"✅ Loading: {TRAIN_CSV_PATH}")
df = pd.read_csv(TRAIN_CSV_PATH)

print("="*60)
print("DATASET OVERVIEW")
print("="*60)
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")

print("\n--- First 5 Rows ---")
display(df.head())

print("\n--- Dataset Info ---")
df.info()

print("\n--- Statistical Summary ---")
display(df.describe())

print("\n--- Target Variable Distribution ---")
print(df['Loan_Status'].value_counts())
print(df['Loan_Status'].value_counts(normalize=True).mul(100).round(2).astype(str) + '%')


## 📖 Cell 5 — Phase 1: Problem Understanding

### Business Problem
Banks and financial institutions receive thousands of loan applications daily. Manually evaluating each application is time-consuming and prone to human bias. A machine learning model can automate this process, providing faster, consistent, and data-driven decisions.

### Why Loan Approval Prediction Matters
- Reduces operational costs for banks
- Provides instant feedback to applicants
- Minimizes risk of loan defaults
- Eliminates human bias in decision-making

### Target Variable
- **`Loan_Status`**: Y (Approved → 1) / N (Rejected → 0)

### Feature Description

| Feature | Type | Description |
|---|---|---|
| Loan_ID | String | Unique loan identifier (dropped) |
| Gender | Categorical | Male / Female |
| Married | Categorical | Applicant married (Yes/No) |
| Dependents | Categorical | Number of dependents (0/1/2/3+) |
| Education | Categorical | Graduate / Not Graduate |
| Self_Employed | Categorical | Self employed (Yes/No) |
| ApplicantIncome | Numerical | Applicant's monthly income |
| CoapplicantIncome | Numerical | Co-applicant's monthly income |
| LoanAmount | Numerical | Loan amount (in thousands) |
| Loan_Amount_Term | Numerical | Term of loan in months |
| Credit_History | Binary | Credit history meets guidelines (1/0) |
| Property_Area | Categorical | Urban / Semi-Urban / Rural |
| **Loan_Status** | **Target** | **Loan approved (Y/N)** |

## 🔧 Cell 6 — Phase 3: Data Preprocessing

In [ ]:
print("="*60)
print("PHASE 3: DATA PREPROCESSING")
print("="*60)

# Step 1: Check missing values
print("\n--- Missing Values Before Cleaning ---")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
print(pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})[missing > 0])

# Step 2: Fill missing values
categorical_cols = ['Gender', 'Married', 'Dependents', 'Self_Employed', 'Credit_History', 'Loan_Amount_Term']
for col in categorical_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)
    print(f"   ✅ Filled '{col}' with mode: {df[col].mode()[0]}")

df['LoanAmount'].fillna(df['LoanAmount'].median(), inplace=True)
print(f"   ✅ Filled 'LoanAmount' with median: {df['LoanAmount'].median()}")

# Step 3: Drop Loan_ID
df.drop('Loan_ID', axis=1, inplace=True)
print("\n   ✅ Dropped 'Loan_ID' column")

# Step 4: Remove duplicates
dup_before = df.duplicated().sum()
df.drop_duplicates(inplace=True)
dup_after = df.duplicated().sum()
print(f"\n   ✅ Duplicates removed: {dup_before} → {dup_after}")

# Step 5: Outlier treatment using IQR capping
def cap_outliers(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    before = df[col].max()
    df[col] = df[col].clip(lower=lower, upper=upper)
    print(f"   ✅ Capped '{col}': max {before:.0f} → {df[col].max():.0f}")

print("\n--- Outlier Capping (IQR Method) ---")
cap_outliers(df, 'LoanAmount')
cap_outliers(df, 'ApplicantIncome')

# Step 6: Encode categorical features
print("\n--- Label Encoding ---")
le = LabelEncoder()
obj_cols = df.select_dtypes(include='object').columns.tolist()
obj_cols = [c for c in obj_cols if c != 'Loan_Status']
for col in obj_cols:
    df[col] = le.fit_transform(df[col])
    print(f"   ✅ Encoded '{col}'")

# Step 7: Encode target variable
df['Loan_Status'] = df['Loan_Status'].map({'Y': 1, 'N': 0})
print("   ✅ Encoded 'Loan_Status': Y→1, N→0")

# Step 8: Confirm no nulls
print("\n--- Post-Preprocessing Check ---")
print(f"   Total nulls remaining: {df.isnull().sum().sum()}")
print(f"   Final shape: {df.shape}")
display(df.head())

## 📊 Cell 7 — Phase 4: Exploratory Data Analysis (EDA)

In [ ]:
print("="*60)
print("PHASE 4: EXPLORATORY DATA ANALYSIS")
print("="*60)

# --- Plot 1: LoanAmount Distribution (before vs after log) ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df['LoanAmount'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('LoanAmount Distribution (Original)', fontsize=13)
axes[0].set_xlabel('Loan Amount'); axes[0].set_ylabel('Frequency')
axes[1].hist(np.log1p(df['LoanAmount']), bins=30, color='darkorange', edgecolor='white')
axes[1].set_title('LoanAmount Distribution (Log Transformed)', fontsize=13)
axes[1].set_xlabel('log(Loan Amount + 1)'); axes[1].set_ylabel('Frequency')
plt.tight_layout(); plt.show()
print("📌 Insight: LoanAmount is right-skewed; log transform makes it approximately normal.")

# --- Plot 2: Loan Status Distribution ---
fig, ax = plt.subplots(figsize=(7, 5))
df['Loan_Status'].value_counts().plot(kind='bar', color=['tomato','steelblue'], edgecolor='white', ax=ax)
ax.set_title('Loan Status Distribution', fontsize=13)
ax.set_xlabel('Loan Status (0=Rejected, 1=Approved)'); ax.set_ylabel('Count')
ax.set_xticklabels(['Approved (1)', 'Rejected (0)'], rotation=0)
for p in ax.patches: ax.annotate(str(p.get_height()), (p.get_x()+0.3, p.get_height()+1))
plt.tight_layout(); plt.show()
print("📌 Insight: Dataset is imbalanced — more approved loans than rejected. SMOTE will address this.")

# --- Plot 3–6: Categorical vs Loan_Status ---
cat_features = ['Gender', 'Married', 'Education', 'Credit_History']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
insights = [
    "Males have higher loan approval counts, but approval rates are similar across genders.",
    "Married applicants show a higher proportion of loan approvals.",
    "Graduates have a higher chance of loan approval than non-graduates.",
    "Credit History is the strongest predictor — applicants with good history are far more likely to be approved."
]
for i, (col, ax) in enumerate(zip(cat_features, axes.flatten())):
    temp = df.groupby([col, 'Loan_Status']).size().unstack(fill_value=0)
    temp.plot(kind='bar', ax=ax, color=['tomato','steelblue'], edgecolor='white', legend=True)
    ax.set_title(f'{col} vs Loan Status', fontsize=12)
    ax.set_xlabel(col); ax.set_ylabel('Count')
    ax.legend(['Rejected (0)','Approved (1)'])
    ax.tick_params(axis='x', rotation=0)
plt.suptitle('Categorical Features vs Loan Status', fontsize=14, y=1.01)
plt.tight_layout(); plt.show()
for col, insight in zip(cat_features, insights):
    print(f"📌 {col}: {insight}")

# --- Plot 7–8: Boxplots ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df.boxplot(column='ApplicantIncome', by='Loan_Status', ax=axes[0], patch_artist=True)
axes[0].set_title('ApplicantIncome by Loan Status'); axes[0].set_xlabel('Loan Status'); axes[0].set_ylabel('Income')
df.boxplot(column='LoanAmount', by='Loan_Status', ax=axes[1], patch_artist=True)
axes[1].set_title('LoanAmount by Loan Status'); axes[1].set_xlabel('Loan Status'); axes[1].set_ylabel('Loan Amount')
plt.suptitle('')
plt.tight_layout(); plt.show()
print("📌 Insight: Income distribution is similar across approval status; loan amount is slightly higher for approved loans.")

# --- Plot 9: Scatter Plot ---
fig, ax = plt.subplots(figsize=(9, 6))
colors = {0: 'tomato', 1: 'steelblue'}
for status, grp in df.groupby('Loan_Status'):
    ax.scatter(grp['ApplicantIncome'], grp['LoanAmount'],
               c=colors[status], label=f"{'Approved' if status==1 else 'Rejected'}", alpha=0.6, s=40)
ax.set_title('ApplicantIncome vs LoanAmount by Loan Status', fontsize=13)
ax.set_xlabel('Applicant Income'); ax.set_ylabel('Loan Amount')
ax.legend(); plt.tight_layout(); plt.show()
print("📌 Insight: No clear linear separation between approved and rejected by income alone — multi-feature model needed.")

# --- Plot 10: Correlation Heatmap ---
fig, ax = plt.subplots(figsize=(12, 9))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', mask=mask,
            linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Heatmap', fontsize=14)
plt.tight_layout(); plt.show()
print("📌 Insight: Credit_History has the highest positive correlation with Loan_Status. LoanAmount and ApplicantIncome are moderately correlated.")

## ⚙️ Cell 8 — Phase 5: Feature Engineering

In [ ]:
print("="*60)
print("PHASE 5: FEATURE ENGINEERING")
print("="*60)

# Step 1: Total Income
df['TotalIncome'] = df['ApplicantIncome'] + df['CoapplicantIncome']
print("✅ Created 'TotalIncome' = ApplicantIncome + CoapplicantIncome")

# Step 2: Log of Loan Amount
df['LoanAmountLog'] = np.log1p(df['LoanAmount'])
print("✅ Created 'LoanAmountLog' = log1p(LoanAmount)")

# Step 3: EMI
df['EMI'] = df['LoanAmount'] / df['Loan_Amount_Term']
print("✅ Created 'EMI' = LoanAmount / Loan_Amount_Term")

# Step 4: Balance Income
df['BalanceIncome'] = df['TotalIncome'] - (df['EMI'] * 1000)
print("✅ Created 'BalanceIncome' = TotalIncome - (EMI × 1000)")

# Step 5: Drop original columns
cols_to_drop = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term']
df.drop(cols_to_drop, axis=1, inplace=True)
print(f"✅ Dropped original columns: {cols_to_drop}")

# Step 6: Final feature list
print(f"\n📋 Final Features ({df.shape[1]-1} features + 1 target):")
for i, col in enumerate(df.columns, 1):
    marker = " ← TARGET" if col == 'Loan_Status' else ""
    print(f"   {i:2}. {col}{marker}")
print(f"\n   Dataset Shape: {df.shape}")

## 🎯 Cell 9 — Feature Selection (SelectKBest)

In [ ]:
print("="*60)
print("FEATURE SELECTION — SelectKBest (chi2)")
print("="*60)

# Step 1: Define X and y
X = df.drop('Loan_Status', axis=1)
y = df['Loan_Status']

# chi2 requires non-negative values — shift negative columns
X_non_neg = X.copy()
for col in X_non_neg.columns:
    if X_non_neg[col].min() < 0:
        X_non_neg[col] = X_non_neg[col] - X_non_neg[col].min()

# Step 2: Apply SelectKBest
k_best = 8
selector = SelectKBest(score_func=chi2, k=k_best)
selector.fit(X_non_neg, y)

# Step 3: Print scores
scores_df = pd.DataFrame({
    'Feature': X.columns,
    'Chi2 Score': selector.scores_
}).sort_values('Chi2 Score', ascending=False)
print("\nChi2 Scores for All Features:")
display(scores_df)

# Step 4: Selected features
selected_mask = selector.get_support()
selected_features = X.columns[selected_mask].tolist()
print(f"\n✅ Top {k_best} Selected Features:")
for f in selected_features:
    print(f"   → {f}")

# Step 5: Apply selection to X
X = X[selected_features]
X_train_cols = selected_features
print(f"\n   X shape after selection: {X.shape}")

## ⚖️ Cell 10 — Data Balancing with SMOTE

In [ ]:
print("="*60)
print("DATA BALANCING — SMOTE (Synthetic Minority Oversampling)")
print("="*60)

# Step 1: Before SMOTE
print("\nClass Distribution BEFORE SMOTE:")
print(y.value_counts())
print(y.value_counts(normalize=True).mul(100).round(2).astype(str) + '%')

# Step 2: Apply SMOTE
smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X, y)

# Step 3: After SMOTE
print("\nClass Distribution AFTER SMOTE:")
print(pd.Series(y_res).value_counts())
print(pd.Series(y_res).value_counts(normalize=True).mul(100).round(2).astype(str) + '%')
print(f"\n✅ Dataset size: {len(y)} → {len(y_res)} samples")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
y.value_counts().plot(kind='bar', ax=axes[0], color=['tomato','steelblue'], edgecolor='white', title='Before SMOTE')
pd.Series(y_res).value_counts().plot(kind='bar', ax=axes[1], color=['steelblue','steelblue'], edgecolor='white', title='After SMOTE')
for ax in axes:
    ax.set_xlabel('Loan Status'); ax.set_ylabel('Count')
    ax.set_xticklabels(['Approved (1)', 'Rejected (0)'], rotation=0)
plt.tight_layout(); plt.show()

## ✂️ Cell 11 — Train-Test Split & Feature Scaling

In [ ]:
print("="*60)
print("TRAIN-TEST SPLIT & FEATURE SCALING")
print("="*60)

# Step 1: Train-test split (80/20, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X_res, y_res, test_size=0.2, random_state=42, stratify=y_res
)

# Step 2: Apply StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Step 3: Print shapes
print(f"\n   X_train: {X_train_scaled.shape}  |  y_train: {y_train.shape}")
print(f"   X_test:  {X_test_scaled.shape}  |  y_test:  {y_test.shape}")
print(f"\n   Train split: {len(y_train)/len(y_res)*100:.1f}%  |  Test split: {len(y_test)/len(y_res)*100:.1f}%")
print("\n✅ Scaling complete — StandardScaler fitted on training data only (no data leakage)")

## 🤖 Cell 12 — Phase 6: Model Building & Training (6 Models)

In [ ]:
%%time
print("="*60)
print("PHASE 6: MODEL TRAINING (6 Models)")
print("="*60)

# Define all 6 models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree":       DecisionTreeClassifier(random_state=42),
    "Random Forest":       RandomForestClassifier(n_estimators=100, random_state=42),
    "SVM":                 SVC(probability=True, random_state=42),
    "KNN":                 KNeighborsClassifier(n_neighbors=5),
    "XGBoost":              XGBClassifier(
                                n_estimators=150, max_depth=5, learning_rate=0.1,
                                use_label_encoder=False, eval_metric='logloss',
                                random_state=42
                            )
}

# Train each model and store predictions
trained_models = {}
predictions    = {}
probabilities  = {}

for name, model in models.items():
    print(f"\n⏳ Training {name}...", end=" ")
    model.fit(X_train_scaled, y_train)
    y_pred      = model.predict(X_test_scaled)
    y_proba     = model.predict_proba(X_test_scaled)[:, 1]
    trained_models[name] = model
    predictions[name]    = y_pred
    probabilities[name]  = y_proba
    print(f"✅ Done! Accuracy: {accuracy_score(y_test, y_pred):.4f}")

print("\n" + "="*60)
print("✅ All 6 models trained successfully!")
print("="*60)

## 📈 Cell 13 — Phase 7: Model Evaluation (6 Models)

In [ ]:
print("="*60)
print("PHASE 7: MODEL EVALUATION")
print("="*60)

results = {}

for name in models.keys():
    y_pred  = predictions[name]
    y_proba = probabilities[name]

    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted')
    rec  = recall_score(y_test, y_pred, average='weighted')
    f1   = f1_score(y_test, y_pred, average='weighted')
    auc  = roc_auc_score(y_test, y_proba)

    results[name] = {'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1 Score': f1, 'ROC-AUC': auc}

    print(f"\n{'='*50}")
    print(f"  MODEL: {name}")
    print(f"{'='*50}")
    print(f"  Accuracy  : {acc:.4f}")
    print(f"  Precision : {prec:.4f}")
    print(f"  Recall    : {rec:.4f}")
    print(f"  F1 Score  : {f1:.4f}")
    print(f"  ROC-AUC   : {auc:.4f}")
    print(f"\n  Classification Report:\n{classification_report(y_test, y_pred)}")

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Rejected','Approved'], yticklabels=['Rejected','Approved'])
    ax.set_title(f'Confusion Matrix — {name}', fontsize=12)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    plt.tight_layout(); plt.show()

# Comparison Table
print("\n" + "="*60)
print("MODEL COMPARISON TABLE")
print("="*60)
results_df = pd.DataFrame(results).T.round(4)
display(results_df.style.highlight_max(color='lightgreen').highlight_min(color='#ffcccc'))

best_acc_model = results_df['Accuracy'].idxmax()
best_auc_model = results_df['ROC-AUC'].idxmax()
print(f"\n🏆 Best Accuracy Model : {best_acc_model} ({results_df.loc[best_acc_model,'Accuracy']:.4f})")
print(f"🏆 Best ROC-AUC Model  : {best_auc_model} ({results_df.loc[best_auc_model,'ROC-AUC']:.4f})")

## 📉 Cell 14 — ROC Curve Comparison (6 Models)

In [ ]:
print("ROC Curve Comparison — All 6 Models")
colors = ['steelblue', 'darkorange', 'green', 'red', 'purple', 'goldenrod']

fig, ax = plt.subplots(figsize=(10, 7))
for (name, y_proba), color in zip(probabilities.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc_score   = roc_auc_score(y_test, y_proba)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC = {auc_score:.4f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Classifier')
ax.set_xlim([0.0, 1.0]); ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve Comparison — All Models', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print("📌 Higher AUC = better model discrimination between approved and rejected loans.")

## 🌳 Cell 15 — Feature Importance (Random Forest)

In [ ]:
rf_model = trained_models['Random Forest']
importances = rf_model.feature_importances_
feat_names  = X_train_cols

feat_df = pd.DataFrame({'Feature': feat_names, 'Importance': importances})\
            .sort_values('Importance', ascending=True).tail(10)

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.barh(feat_df['Feature'], feat_df['Importance'], color='steelblue', edgecolor='white')
ax.set_xlabel('Feature Importance Score', fontsize=12)
ax.set_title('Top 10 Feature Importances — Random Forest', fontsize=14, fontweight='bold')
for bar, val in zip(bars, feat_df['Importance']):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)
plt.tight_layout(); plt.show()
print("📌 Credit_History typically ranks as the most important feature for loan approval.")

## 🔬 Cell 16 — Hyperparameter Tuning (GridSearchCV)

In [ ]:
%%time
print("="*60)
print("HYPERPARAMETER TUNING — GridSearchCV")
print("="*60)

# Tune the best AUC model among all 6
best_auc_model_name = results_df['ROC-AUC'].idxmax()
print(f"\nTuning model: {best_auc_model_name}")

param_grids = {
    'Random Forest': {
        'n_estimators': [50, 100, 200],
        'max_depth': [None, 5, 10],
        'min_samples_split': [2, 5]
    },
    'Logistic Regression': {
        'C': [0.01, 0.1, 1, 10],
        'solver': ['lbfgs', 'liblinear']
    },
    'Decision Tree': {
        'max_depth': [None, 5, 10, 15],
        'min_samples_split': [2, 5, 10]
    },
    'SVM': {
        'C': [0.1, 1, 10],
        'kernel': ['rbf', 'linear']
    },
    'KNN': {
        'n_neighbors': [3, 5, 7, 11],
        'weights': ['uniform', 'distance']
    },
    'XGBoost': {
        'n_estimators': [100, 150, 200],
        'max_depth': [3, 5, 7],
        'learning_rate': [0.05, 0.1, 0.2]
    }
}

base_model = trained_models[best_auc_model_name]
param_grid = param_grids[best_auc_model_name]

# Re-instantiate a fresh copy of the best model class for clean tuning
extra_kwargs = {}
if 'random_state' in base_model.get_params():
    extra_kwargs['random_state'] = 42
if best_auc_model_name == 'XGBoost':
    extra_kwargs['use_label_encoder'] = False
    extra_kwargs['eval_metric'] = 'logloss'
if best_auc_model_name == 'SVM':
    extra_kwargs['probability'] = True

grid_search = GridSearchCV(
    estimator=base_model.__class__(**extra_kwargs),
    param_grid=param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train_scaled, y_train)

print(f"\n✅ Best Parameters: {grid_search.best_params_}")
print(f"✅ Best CV ROC-AUC : {grid_search.best_score_:.4f}")

# Final evaluation with best model
best_model = grid_search.best_estimator_
y_pred_best  = best_model.predict(X_test_scaled)
y_proba_best = best_model.predict_proba(X_test_scaled)[:, 1]

print(f"\n--- Final Tuned Model Evaluation ---")
print(f"  Accuracy : {accuracy_score(y_test, y_pred_best):.4f}")
print(f"  F1 Score : {f1_score(y_test, y_pred_best, average='weighted'):.4f}")
print(f"  ROC-AUC  : {roc_auc_score(y_test, y_proba_best):.4f}")

## 💾 Cell 17 — Save the Best Model

In [ ]:
from google.colab import files

# Step 1: Save model, scaler, selector, feature names
joblib.dump(best_model, 'loan_model.pkl')
joblib.dump(scaler,     'scaler.pkl')
joblib.dump(selector,   'selector.pkl')

with open('feature_names.json', 'w') as f:
    json.dump(X_train_cols, f)

print("✅ Saved: loan_model.pkl")
print("✅ Saved: scaler.pkl")
print("✅ Saved: selector.pkl")
print("✅ Saved: feature_names.json")

# Step 2: Download all 4 files
print("\nDownloading files...")
files.download('loan_model.pkl')
files.download('scaler.pkl')
files.download('selector.pkl')
files.download('feature_names.json')
print("\n🎉 All files downloaded! Place them in the same folder as app.py for Streamlit deployment.")

## 📝 Cell 18 — Summary & Conclusion

### Dataset Summary
- **Source:** Kaggle Loan Prediction Dataset (altruistdelhite04)
- **Original Size:** ~614 rows × 13 columns
- **After SMOTE:** Balanced dataset with equal approved/rejected samples
- **Features Used:** 8 selected features (after engineering + SelectKBest)

### Models Trained (6 Total)
| Model | Type |
|---|---|
| Logistic Regression | Linear Classifier |
| Decision Tree | Tree-based |
| Random Forest | Ensemble (Bagging) |
| SVM | Kernel-based |
| KNN | Instance-based |
| XGBoost | Ensemble (Boosting) |

### Key EDA Insights
1. **Credit History** is the most important predictor — applicants with good credit history have ~80% approval rate
2. **LoanAmount** is right-skewed — log transformation normalizes it
3. **Married graduates** from **Semi-Urban** areas have the highest approval rates
4. The dataset was imbalanced (422 approved vs 192 rejected) — fixed with SMOTE
5. **XGBoost** and **Random Forest** typically edge out the other models on ROC-AUC due to their ensemble nature

### Limitations
- Dataset is relatively small (~614 samples)
- Synthetic dataset may not perfectly capture real-world loan patterns
- Model does not account for macroeconomic factors

### Future Scope
- Integrate real-time credit score APIs
- Add SHAP explainability for prediction transparency
- Deploy as a REST API with authentication
- Expand with deep learning models (neural networks)
- Add applicant risk scoring dashboard

---
*AIML Summer Internship 2026 · IIHMF, MNNIT Allahabad, Prayagraj*